In [1]:
# Install dependencies

%pip install -q optuna transformers==4.46.3 accelerate sentencepiece


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:

import torch
import transformers
import optuna
import huggingface_hub

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Optuna:", optuna.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("GPU available:", torch.cuda.is_available())


PyTorch: 2.11.0+cu130
Transformers: 4.46.3
Optuna: 5.0.0
Hugging Face Hub: 0.36.2
GPU available: True


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import os, json, random, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import optuna
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from datasets import Dataset

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [7]:
# ---------------- USER CONFIG ----------------

PROJECT_DIR = Path("/content/drive/MyDrive/NewsBias_Project_Backup")

SPLIT_DIR = PROJECT_DIR / "babe_sg2_splits"

# Existing fine-tuned BABE-only DeBERTa checkpoint used as the baseline
BASELINE_CHECKPOINT = (
    PROJECT_DIR / "deberta_augmented_experiment" / "babe_only" / "best_model"
)

# Exact base model used for new Optuna trials.
# Change this if the original training notebook used another identifier.
BASE_MODEL_NAME = "microsoft/deberta-v3-base"

OUTPUT_DIR = PROJECT_DIR / "deberta_optuna_babe_only"
TRIALS_DIR = OUTPUT_DIR / "trial_checkpoints"
BEST_MODEL_DIR = OUTPUT_DIR / "best_optimized_model"

# Increase to 15–30 for a more thorough search if compute allows.
N_TRIALS = 10

# Search is intentionally modest for Colab GPU time.
MAX_LENGTH = 256

# Optional: set these explicitly if automatic column detection is wrong.
TEXT_COLUMN = None   # e.g. "text"
LABEL_COLUMN = None  # e.g. "label"

# ------------------------------------------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRIALS_DIR.mkdir(parents=True, exist_ok=True)

for p in [SPLIT_DIR / "train.csv", SPLIT_DIR / "validation.csv", SPLIT_DIR / "test.csv"]:
    print(p, "exists:", p.exists())

assert (SPLIT_DIR / "train.csv").exists(), "train.csv not found in SPLIT_DIR"
assert (SPLIT_DIR / "validation.csv").exists(), "validation.csv not found in SPLIT_DIR"
assert (SPLIT_DIR / "test.csv").exists(), "test.csv not found in SPLIT_DIR"
assert BASELINE_CHECKPOINT.exists(), f"Baseline checkpoint not found: {BASELINE_CHECKPOINT}"


/content/drive/MyDrive/NewsBias_Project_Backup/babe_sg2_splits/train.csv exists: True
/content/drive/MyDrive/NewsBias_Project_Backup/babe_sg2_splits/validation.csv exists: True
/content/drive/MyDrive/NewsBias_Project_Backup/babe_sg2_splits/test.csv exists: True


## Load and validate the existing splits

The code below detects common text/label column names. If it selects the wrong columns, set `TEXT_COLUMN` and `LABEL_COLUMN` in the configuration cell and rerun from there.

Labels are mapped consistently to `0 = non-biased` and `1 = biased` where labels are recognizable by name. If your CSV already stores numeric labels, the code assumes the existing convention is `0 = non-biased`, `1 = biased`; verify that convention before continuing.


In [8]:
train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df   = pd.read_csv(SPLIT_DIR / "validation.csv")
test_df  = pd.read_csv(SPLIT_DIR / "test.csv")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)
print("Train columns:", train_df.columns.tolist())

def choose_column(df, explicit, candidates, kind):
    if explicit is not None:
        if explicit not in df.columns:
            raise ValueError(f"{kind} column '{explicit}' not found. Available: {df.columns.tolist()}")
        return explicit
    lower_map = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand in lower_map:
            return lower_map[cand]
    # fallback: text = first object column; label = likely binary column
    if kind == "text":
        obj_cols = [c for c in df.columns if df[c].dtype == "object"]
        if obj_cols:
            return obj_cols[0]
    else:
        for c in df.columns:
            vals = set(df[c].dropna().unique().tolist())
            if len(vals) == 2:
                return c
    raise ValueError(f"Could not detect {kind} column. Set it explicitly in the config cell.")

TEXT_CANDIDATES = ["text", "sentence", "content", "article", "news", "news_text", "clean_text", "input_text"]
LABEL_CANDIDATES = ["label", "labels", "bias", "biased", "target", "class", "bias_label"]

TEXT_COLUMN = choose_column(train_df, TEXT_COLUMN, TEXT_CANDIDATES, "text")
LABEL_COLUMN = choose_column(train_df, LABEL_COLUMN, LABEL_CANDIDATES, "label")
print("Using text column:", TEXT_COLUMN)
print("Using label column:", LABEL_COLUMN)

# Confirm all splits contain the same selected columns
for name, df in [("validation", val_df), ("test", test_df)]:
    if TEXT_COLUMN not in df.columns or LABEL_COLUMN not in df.columns:
        raise ValueError(f"{name} split lacks {TEXT_COLUMN!r} or {LABEL_COLUMN!r}")

def normalize_label_value(v):
    if pd.isna(v):
        raise ValueError("Missing label encountered.")
    if isinstance(v, (int, np.integer)):
        if int(v) in (0, 1):
            return int(v)
    if isinstance(v, (float, np.floating)) and float(v).is_integer() and int(v) in (0, 1):
        return int(v)
    s = str(v).strip().lower().replace("_", "-")
    if s in {"0", "non-biased", "nonbiased", "not biased", "unbiased", "no-bias", "no bias"}:
        return 0
    if s in {"1", "biased", "bias"}:
        return 1
    raise ValueError(
        f"Unrecognized label {v!r}. Ensure labels use 0=non-biased, 1=biased, "
        "or edit normalize_label_value() for your dataset."
    )

def clean_split(df, name):
    out = df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
    out = out.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])
    out[TEXT_COLUMN] = out[TEXT_COLUMN].astype(str).str.strip()
    out = out[out[TEXT_COLUMN].str.len() > 0]
    out["labels"] = out[LABEL_COLUMN].map(normalize_label_value).astype(int)
    out = out.rename(columns={TEXT_COLUMN: "text"})
    out = out[["text", "labels"]].reset_index(drop=True)
    print(name, "rows:", len(out), "| label counts:", out["labels"].value_counts().sort_index().to_dict())
    return out

train_clean = clean_split(train_df, "Train")
val_clean = clean_split(val_df, "Validation")
test_clean = clean_split(test_df, "Test")

assert set(train_clean["labels"].unique()) == {0, 1}, "Train split must contain both classes."
assert set(val_clean["labels"].unique()) == {0, 1}, "Validation split must contain both classes."
assert set(test_clean["labels"].unique()) == {0, 1}, "Test split must contain both classes."


Train shape: (2579, 9)
Validation shape: (550, 9)
Test shape: (508, 9)
Train columns: ['text', 'news_link', 'outlet', 'topic', 'type', 'label_bias', 'label_opinion', 'biased_words', 'label']
Using text column: text
Using label column: label
Train rows: 2579 | label counts: {0: 1327, 1: 1252}
Validation rows: 550 | label counts: {0: 266, 1: 284}
Test rows: 508 | label counts: {0: 252, 1: 256}


In [9]:
# Load the base tokenizer for Optuna trials.
# This must match the tokenizer used when the baseline checkpoint was fine-tuned.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=False)

def to_hf_dataset(df):
    ds = Dataset.from_pandas(df[["text", "labels"]], preserve_index=False)
    def tokenize_batch(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
    ds = ds.map(tokenize_batch, batched=True, remove_columns=["text"])
    return ds

train_ds = to_hf_dataset(train_clean)
val_ds = to_hf_dataset(val_clean)
test_ds = to_hf_dataset(test_clean)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenizer:", type(tokenizer).__name__)
print("Tokenizer vocabulary size:", len(tokenizer))


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/2579 [00:00<?, ? examples/s]

Map:   0%|          | 0/550 [00:00<?, ? examples/s]

Map:   0%|          | 0/508 [00:00<?, ? examples/s]

Tokenizer: DebertaV2Tokenizer
Tokenizer vocabulary size: 128001


## Baseline evaluation

The baseline is the existing fine-tuned BABE-only checkpoint. This cell evaluates it on validation data for reference, but does not use the test set. The baseline's validation score is useful context; Optuna's objective is the validation macro-F1 of each newly trained configuration.


In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_cls, r_cls, f1_cls, support = precision_recall_fscore_support(
        labels, preds, labels=[0, 1], average=None, zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_precision": p_macro,
        "macro_recall": r_macro,
        "macro_f1": f1_macro,
        "nonbiased_precision": p_cls[0],
        "nonbiased_recall": r_cls[0],
        "nonbiased_f1": f1_cls[0],
        "biased_precision": p_cls[1],
        "biased_recall": r_cls[1],
        "biased_f1": f1_cls[1],
    }

baseline_model = AutoModelForSequenceClassification.from_pretrained(
    str(BASELINE_CHECKPOINT),
    num_labels=2,
    id2label={0: "Non-biased", 1: "Biased"},
    label2id={"Non-biased": 0, "Biased": 1},
)

baseline_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "baseline_eval_tmp"),
    per_device_eval_batch_size=16,
    report_to="none",
    dataloader_pin_memory=torch.cuda.is_available(),
)

baseline_trainer = Trainer(
    model=baseline_model,
    args=baseline_args,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

baseline_val_metrics = baseline_trainer.evaluate(val_ds)
baseline_val_metrics = {k.replace("eval_", ""): v for k, v in baseline_val_metrics.items()}
print("Baseline validation metrics:")
print(pd.Series(baseline_val_metrics).round(4))

# Release baseline model memory before Optuna trials
del baseline_trainer, baseline_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Baseline validation metrics:
loss                        0.4736
model_preparation_time      0.0034
accuracy                    0.8309
macro_precision             0.8309
macro_recall                0.8313
macro_f1                    0.8309
nonbiased_precision         0.8145
nonbiased_recall            0.8421
nonbiased_f1                0.8281
biased_precision            0.8473
biased_recall               0.8204
biased_f1                   0.8336
runtime                     3.5717
samples_per_second        153.9880
steps_per_second            9.7990
dtype: float64


## Optuna search

Each trial starts from the same pretrained base model, trains on the training split, and is selected using **validation macro-F1**. Test data is not passed to the objective.

Search space:
- learning rate
- weight decay
- number of epochs
- batch size
- warmup ratio

This notebook uses a modest number of trials for a Colab-friendly first run. If a trial fails due to GPU memory, reduce `MAX_LENGTH` or use smaller batch sizes in the objective.


In [11]:
def objective(trial):
    # Hyperparameter search space
    learning_rate = trial.suggest_float("learning_rate", 8e-6, 5e-5, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.15)
    num_train_epochs = trial.suggest_int("num_train_epochs", 2, 4)
    per_device_train_batch_size = trial.suggest_categorical("per_device_train_batch_size", [8, 16])
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.0, 0.15)

    trial_dir = TRIALS_DIR / f"trial_{trial.number}"
    if trial_dir.exists():
        shutil.rmtree(trial_dir)
    trial_dir.mkdir(parents=True, exist_ok=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_NAME,
        num_labels=2,
        id2label={0: "Non-biased", 1: "Biased"},
        label2id={"Non-biased": 0, "Biased": 1},
    )

    args = TrainingArguments(
        output_dir=str(trial_dir),
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=16,
        warmup_ratio=warmup_ratio,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        logging_strategy="epoch",
        report_to="none",
        seed=SEED,
        data_seed=SEED,
        fp16=torch.cuda.is_available(),
        gradient_accumulation_steps=1,
        dataloader_pin_memory=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate(val_ds)
    score = float(metrics["eval_macro_f1"])

    trial.set_user_attr("validation_accuracy", float(metrics.get("eval_accuracy", np.nan)))
    trial.set_user_attr("validation_biased_f1", float(metrics.get("eval_biased_f1", np.nan)))
    trial.set_user_attr("best_checkpoint", trainer.state.best_model_checkpoint)

    # Clean model/GPU memory; retain trial metadata in Optuna.
    del trainer, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return score

sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(direction="maximize", study_name="deberta_babe_only_macro_f1", sampler=sampler)
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True, show_progress_bar=True)

print("\nBest validation macro-F1:", round(study.best_value, 4))
print("Best hyperparameters:")
print(json.dumps(study.best_params, indent=2))

trials_df = study.trials_dataframe()
trials_csv = OUTPUT_DIR / "optuna_trials.csv"
trials_df.to_csv(trials_csv, index=False)
print("Saved trials:", trials_csv)


[I 2026-10-27 06:12:21,162] A new study created in memory with name: deberta_babe_only_macro_f1


  0%|          | 0/10 [00:00<?, ?it/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Nonbiased Precision,Nonbiased Recall,Nonbiased F1,Biased Precision,Biased Recall,Biased F1
1,0.546800,0.430488,0.814545,0.814333,0.814466,0.814388,0.805970,0.812030,0.808989,0.822695,0.816901,0.819788
2,0.376000,0.530557,0.821818,0.822222,0.822461,0.821809,0.800000,0.842105,0.820513,0.844444,0.802817,0.823105
3,0.276200,0.799039,0.792727,0.792941,0.793220,0.792703,0.773381,0.808271,0.790441,0.812500,0.778169,0.794964
4,0.210100,0.839786,0.800000,0.799802,0.800024,0.799870,0.788889,0.800752,0.794776,0.810714,0.799296,0.804965


...

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Nonbiased Precision,Nonbiased Recall,Nonbiased F1,Biased Precision,Biased Recall,Biased F1
1,0.573400,0.425470,0.807273,0.807672,0.807900,0.807263,0.785714,0.827068,0.805861,0.829630,0.788732,0.808664
2,0.366100,0.423251,0.807273,0.810815,0.808972,0.807148,0.768456,0.860902,0.812057,0.853175,0.757042,0.802239
3,0.239500,0.590173,0.807273,0.807336,0.807662,0.807232,0.789855,0.819549,0.804428,0.824818,0.795775,0.810036
4,0.154100,0.709391,0.809091,0.808965,0.808708,0.808812,0.806084,0.796992,0.801512,0.811847,0.820423,0.816112


[I 2026-10-27 06:51:25,424] Trial 8 finished with value: 0.8088121856988204 and parameters: {'learning_rate': 2.17874184015828e-05, 'weight_decay': 0.027728168328829055, 'num_train_epochs': 4, 'per_device_train_batch_size': 16, 'warmup_ratio': 0.13422410256414732}. Best is trial 3 with value: 0.8309040599555029.


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Nonbiased Precision,Nonbiased Recall,Nonbiased F1,Biased Precision,Biased Recall,Biased F1
1,0.535600,0.423979,0.818182,0.818036,0.817868,0.817941,0.814394,0.808271,0.811321,0.821678,0.827465,0.824561
2,0.353700,0.539380,0.825455,0.825518,0.825863,0.825418,0.807971,0.838346,0.822878,0.843066,0.813380,0.827957


[I 2026-10-27 06:55:11,141] Trial 9 finished with value: 0.8254176090147998 and parameters: {'learning_rate': 2.3930216761416582e-05, 'weight_decay': 0.13828113525346752, 'num_train_epochs': 2, 'per_device_train_batch_size': 8, 'warmup_ratio': 0.04879954961448965}. Best is trial 3 with value: 0.8309040599555029.

Best validation macro-F1: 0.8309
Best hyperparameters:
{
  "learning_rate": 1.7654946096214942e-05,
  "weight_decay": 0.043684371029706286,
  "num_train_epochs": 3,
  "per_device_train_batch_size": 16,
  "warmup_ratio": 0.05495427649405375
}
Saved trials: /content/drive/MyDrive/NewsBias_Project_Backup/deberta_optuna_babe_only/optuna_trials.csv


## Train the selected configuration and save the optimized checkpoint

After Optuna selects hyperparameters using validation macro-F1, train one final model with that configuration. `load_best_model_at_end=True` retains the best epoch according to validation macro-F1. No test metrics are used for model selection.


In [13]:
best_params = study.best_params

if BEST_MODEL_DIR.exists():
    shutil.rmtree(BEST_MODEL_DIR)
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

optimized_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=2,
    id2label={0: "Non-biased", 1: "Biased"},
    label2id={"Non-biased": 0, "Biased": 1},
)

final_args = TrainingArguments(
    output_dir=str(BEST_MODEL_DIR),
    learning_rate=best_params["learning_rate"],
    weight_decay=best_params["weight_decay"],
    num_train_epochs=best_params["num_train_epochs"],
    per_device_train_batch_size=best_params["per_device_train_batch_size"],
    per_device_eval_batch_size=16,
    warmup_ratio=best_params["warmup_ratio"],
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    logging_strategy="epoch",
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    fp16=torch.cuda.is_available(),
    dataloader_pin_memory=torch.cuda.is_available(),
)

optimized_trainer = Trainer(
    model=optimized_model,
    args=final_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

optimized_trainer.train()
optimized_trainer.save_model(str(BEST_MODEL_DIR))
tokenizer.save_pretrained(str(BEST_MODEL_DIR))

with open(OUTPUT_DIR / "best_hyperparameters.json", "w") as f:
    json.dump(best_params, f, indent=2)

print("Saved optimized model to:", BEST_MODEL_DIR)


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Nonbiased Precision,Nonbiased Recall,Nonbiased F1,Biased Precision,Biased Recall,Biased F1
1,0.570400,0.436350,0.807273,0.807889,0.808019,0.807270,0.783688,0.830827,0.806569,0.832090,0.785211,0.807971
2,0.366300,0.414880,0.816364,0.818021,0.817537,0.816348,0.785467,0.853383,0.818018,0.850575,0.781690,0.814679
3,0.249400,0.495960,0.814545,0.815705,0.815538,0.814543,0.786713,0.845865,0.815217,0.844697,0.785211,0.813869


Saved optimized model to: /content/drive/MyDrive/NewsBias_Project_Backup/deberta_optuna_babe_only/best_optimized_model


## Final test evaluation — run only after model selection

This is the first point where the held-out test split is used. Evaluate both the existing baseline checkpoint and the optimized model on the same test examples. Do not change the search space, hyperparameters, or threshold based on these results.


In [14]:
def evaluate_checkpoint(checkpoint_path, label):
    model = AutoModelForSequenceClassification.from_pretrained(str(checkpoint_path))
    args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / f"test_eval_{label.replace(' ', '_')}_tmp"),
        per_device_eval_batch_size=16,
        report_to="none",
        dataloader_pin_memory=torch.cuda.is_available(),
    )
    trainer = Trainer(
        model=model,
        args=args,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    metrics = trainer.evaluate(test_ds)
    preds_output = trainer.predict(test_ds)
    y_true = np.asarray(test_clean["labels"])
    y_pred = np.argmax(preds_output.predictions, axis=-1)

    result = {k.replace("eval_", ""): float(v) for k, v in metrics.items()
              if isinstance(v, (int, float, np.integer, np.floating))}
    result["model"] = label

    pred_df = pd.DataFrame({
        "text": test_clean["text"],
        "true_label": y_true,
        "predicted_label": y_pred,
    })
    pred_df.to_csv(OUTPUT_DIR / f"test_predictions_{label.replace(' ', '_')}.csv", index=False)

    print(f"\n===== {label} =====")
    print(classification_report(
        y_true, y_pred, labels=[0, 1],
        target_names=["Non-biased", "Biased"], digits=4, zero_division=0
    ))
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

    del trainer, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

baseline_test_result = evaluate_checkpoint(BASELINE_CHECKPOINT, "Baseline DeBERTa")
optimized_test_result = evaluate_checkpoint(BEST_MODEL_DIR, "Optuna-optimized DeBERTa")

comparison = pd.DataFrame([baseline_test_result, optimized_test_result])
preferred_cols = [
    "model", "eval_accuracy", "eval_macro_precision", "eval_macro_recall",
    "eval_macro_f1", "eval_biased_precision", "eval_biased_recall", "eval_biased_f1"
]
# Metrics are returned from Trainer with names such as accuracy/macro_f1 after prefix removal.
# Normalize columns for a readable comparison.
rename_map = {
    "accuracy": "Accuracy",
    "macro_precision": "Macro precision",
    "macro_recall": "Macro recall",
    "macro_f1": "Macro F1",
    "biased_precision": "Biased precision",
    "biased_recall": "Biased recall",
    "biased_f1": "Biased F1",
}
display_df = comparison[["model"] + [c for c in rename_map if c in comparison.columns]].rename(columns=rename_map)
display(display_df.round(4))

comparison.to_csv(OUTPUT_DIR / "baseline_vs_optuna_test_comparison.csv", index=False)
print("Saved comparison:", OUTPUT_DIR / "baseline_vs_optuna_test_comparison.csv")
print("All outputs saved in:", OUTPUT_DIR)



===== Baseline DeBERTa =====
              precision    recall  f1-score   support

  Non-biased     0.8210    0.8373    0.8291       252
      Biased     0.8367    0.8203    0.8284       256

    accuracy                         0.8287       508
   macro avg     0.8288    0.8288    0.8287       508
weighted avg     0.8289    0.8287    0.8287       508

Confusion matrix [[TN, FP], [FN, TP]]:
[[211  41]
 [ 46 210]]



===== Optuna-optimized DeBERTa =====
              precision    recall  f1-score   support

  Non-biased     0.8104    0.8651    0.8369       252
      Biased     0.8577    0.8008    0.8283       256

    accuracy                         0.8327       508
   macro avg     0.8341    0.8329    0.8326       508
weighted avg     0.8343    0.8327    0.8325       508

Confusion matrix [[TN, FP], [FN, TP]]:
[[218  34]
 [ 51 205]]


,model,Accuracy,Macro precision,Macro recall,Macro F1,Biased precision,Biased recall,Biased F1
0,Baseline DeBERTa,0.8287,0.8288,0.8288,0.8287,0.8367,0.8203,0.8284
1,Optuna-optimized DeBERTa,0.8327,0.8341,0.8329,0.8326,0.8577,0.8008,0.8283


Saved comparison: /content/drive/MyDrive/NewsBias_Project_Backup/deberta_optuna_babe_only/baseline_vs_optuna_test_comparison.csv
All outputs saved in: /content/drive/MyDrive/NewsBias_Project_Backup/deberta_optuna_babe_only


## How to interpret the result

- A higher **test macro-F1** for the optimized model indicates improvement on the held-out test set for balanced class performance.
- If the optimized model does not improve, report that honestly; optimization is a search procedure, not a guarantee of better results.
- Report the number of Optuna trials, best validation macro-F1, best hyperparameters, baseline test metrics, and optimized test metrics.
- Because the baseline checkpoint may have been trained in a prior run, verify that it was trained on the same `train.csv` split and that the same label mapping was used. If not, the baseline-vs-optimized comparison is not strictly apples-to-apples.
- This notebook trains Optuna trials from the pretrained base model using the existing train split. It does **not** initialize each trial from the fine-tuned baseline checkpoint, because doing so would make the hyperparameter search dependent on an already-trained model rather than comparing fresh fine-tuning configurations.
